# SKKU merged YOLOv8 segmentation training

기존 차선/신호 데이터와 obstacle 추가 데이터를 병합한 COCO segmentation ZIP을 YOLO 형식으로 변환하고 학습합니다.

- 병합 클래스: `arrow`, `crosswalk`, `lane-center`, `lane-side`, `light`, `obstacle`
- 병합 폴더 `skku_merged_obstacle_coco` 또는 같은 이름의 ZIP을 Google Drive의 `/content/drive/MyDrive/차온우/`에 둔 뒤 위에서부터 순서대로 실행합니다.
- 폴리곤과 RLE 형식의 COCO segmentation을 모두 변환합니다.


## 1. Colab 환경 설정


In [ ]:
!nvidia-smi
!pip install -q "ultralytics>=8.3,<9" pycocotools


In [ ]:
import ultralytics

print("Ultralytics:", ultralytics.__version__)
ultralytics.checks()


## 2. Google Drive와 데이터 경로 설정


In [ ]:
import shutil
from pathlib import Path
from zipfile import ZipFile

from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/차온우")
DATASET_NAME = "skku_merged_obstacle_coco"
MERGED_ZIP = DRIVE_DIR / f"{DATASET_NAME}.zip"
DRIVE_COCO_DIR = DRIVE_DIR / DATASET_NAME

WORK_DIR = Path("/content/skku_training")
LOCAL_COCO_PARENT = WORK_DIR / "coco"
COCO_DIR = LOCAL_COCO_PARENT / DATASET_NAME
YOLO_DIR = WORK_DIR / "skku_merged_yolo_seg"
RUNS_DIR = Path("/content/runs")
RUN_NAME = "yolov8n_seg_merged_aug"


def copy_directory_with_progress(source, target):
    files = [path for path in source.rglob("*") if path.is_file()]
    for source_file in tqdm(files, desc="Copy COCO dataset", unit="file"):
        target_file = target / source_file.relative_to(source)
        target_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source_file, target_file)


def coco_dataset_ready(path):
    return all(
        (path / split / "_annotations.coco.json").is_file()
        for split in ("train", "valid", "test")
    )


if not coco_dataset_ready(COCO_DIR):
    if COCO_DIR.exists():
        shutil.rmtree(COCO_DIR)
    LOCAL_COCO_PARENT.mkdir(parents=True, exist_ok=True)
    if MERGED_ZIP.is_file():
        with ZipFile(MERGED_ZIP) as archive:
            for member in tqdm(
                archive.infolist(), desc="Extract COCO ZIP", unit="file"
            ):
                archive.extract(member, LOCAL_COCO_PARENT)
    elif DRIVE_COCO_DIR.is_dir():
        copy_directory_with_progress(DRIVE_COCO_DIR, COCO_DIR)
    else:
        raise FileNotFoundError(
            f"병합 ZIP을 찾을 수 없습니다: {MERGED_ZIP}. "
            "skku_merged_obstacle_coco.zip을 이 위치에 업로드하세요."
        )

if not coco_dataset_ready(COCO_DIR):
    raise RuntimeError(f"압축 해제 후 데이터셋을 확인할 수 없습니다: {COCO_DIR}")

print("Local COCO dataset:", COCO_DIR)
print("YOLO output:", YOLO_DIR)


## 3. 병합 데이터 검증


In [ ]:
import json
from collections import Counter

SPLITS = ("train", "valid", "test")


def load_coco(split):
    annotation_path = COCO_DIR / split / "_annotations.coco.json"
    if not annotation_path.is_file():
        raise FileNotFoundError(annotation_path)
    return json.loads(annotation_path.read_text(encoding="utf-8"))


split_summary = {}
reference_categories = None
for split in SPLITS:
    coco = load_coco(split)
    categories = tuple(
        item["name"] for item in sorted(coco["categories"], key=lambda item: item["id"])
    )
    if reference_categories is None:
        reference_categories = categories
    elif categories != reference_categories:
        raise ValueError(f"{split}의 클래스 순서가 다른 split과 다릅니다.")

    image_ids = {item["id"] for item in coco["images"]}
    category_names = {item["id"]: item["name"] for item in coco["categories"]}
    missing_files = [
        item["file_name"]
        for item in coco["images"]
        if not (COCO_DIR / split / item["file_name"]).is_file()
    ]
    invalid_references = [
        item["id"] for item in coco["annotations"] if item["image_id"] not in image_ids
    ]
    if missing_files or invalid_references:
        raise ValueError(
            f"{split}: missing_files={len(missing_files)}, "
            f"invalid_references={len(invalid_references)}"
        )

    counts = Counter(category_names[item["category_id"]] for item in coco["annotations"])
    split_summary[split] = {
        "images": len(coco["images"]),
        "annotations": len(coco["annotations"]),
        "classes": dict(sorted(counts.items())),
    }

if "obstacle" not in reference_categories:
    raise ValueError("병합 데이터에 obstacle 클래스가 없습니다.")

print("classes:", list(reference_categories))
print(json.dumps(split_summary, ensure_ascii=False, indent=2))


## 4. COCO segmentation을 YOLO segmentation으로 변환

COCO polygon은 그대로 정규화하고, 압축 RLE mask는 contour로 변환합니다. 진행률이 표시되며, 변환이 이미 끝난 경우 기존 결과를 재사용합니다. 처음부터 다시 만들 때만 `FORCE_REBUILD = True`로 바꾸세요.


In [ ]:
import shutil
from collections import defaultdict

import cv2
import numpy as np
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm


def decode_rle(segmentation, height, width):
    rle = dict(segmentation)
    counts = rle.get("counts")
    if isinstance(counts, list):
        rle = mask_utils.frPyObjects(rle, height, width)
    elif isinstance(counts, str):
        rle["counts"] = counts.encode("ascii")

    mask = mask_utils.decode(rle)
    if mask.ndim == 3:
        mask = np.any(mask, axis=2)
    return mask.astype(np.uint8)


def segmentation_to_polygons(segmentation, height, width):
    if isinstance(segmentation, dict):
        mask = decode_rle(segmentation, height, width)
        contours, _ = cv2.findContours(
            mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        polygons = [
            contour.reshape(-1, 2).astype(np.float32)
            for contour in contours
            if len(contour) >= 3 and cv2.contourArea(contour) >= 2.0
        ]
        return sorted(polygons, key=cv2.contourArea, reverse=True)

    if not isinstance(segmentation, list) or not segmentation:
        return []

    raw_polygons = segmentation
    if isinstance(segmentation[0], (int, float)):
        raw_polygons = [segmentation]

    polygons = []
    for raw_polygon in raw_polygons:
        if len(raw_polygon) < 6 or len(raw_polygon) % 2:
            continue
        polygon = np.asarray(raw_polygon, dtype=np.float32).reshape(-1, 2)
        if np.isfinite(polygon).all() and len(np.unique(polygon, axis=0)) >= 3:
            polygons.append(polygon)
    return polygons


def polygon_to_yolo_line(class_id, polygon, width, height):
    normalized = polygon.copy()
    normalized[:, 0] = np.clip(normalized[:, 0] / width, 0.0, 1.0)
    normalized[:, 1] = np.clip(normalized[:, 1] / height, 0.0, 1.0)
    coordinates = " ".join(f"{value:.6f}" for value in normalized.reshape(-1))
    return f"{class_id} {coordinates}"


def load_existing_conversion(yolo_dir):
    data_yaml = yolo_dir / "data.yaml"
    summary_path = yolo_dir / "conversion_summary.json"
    if not data_yaml.is_file() or not summary_path.is_file():
        return None

    payload = json.loads(summary_path.read_text(encoding="utf-8"))
    print("기존 YOLO 변환 결과를 재사용합니다:", yolo_dir)
    return data_yaml, payload["class_names"], payload["splits"]


def convert_coco_to_yolo(coco_dir, yolo_dir, force_rebuild=False):
    if not force_rebuild:
        existing = load_existing_conversion(yolo_dir)
        if existing is not None:
            return existing

    if yolo_dir.exists():
        shutil.rmtree(yolo_dir)

    first_coco = load_coco("train")
    categories = sorted(first_coco["categories"], key=lambda item: item["id"])
    class_names = [item["name"] for item in categories]
    expected_mapping = {item["name"]: index for index, item in enumerate(categories)}
    conversion_summary = {}

    for split in SPLITS:
        coco = load_coco(split)
        split_mapping = {
            item["id"]: expected_mapping[item["name"]] for item in coco["categories"]
        }
        images_by_id = {item["id"]: item for item in coco["images"]}
        annotations_by_image = defaultdict(list)
        for annotation in coco["annotations"]:
            annotations_by_image[annotation["image_id"]].append(annotation)

        image_output = yolo_dir / "images" / split
        label_output = yolo_dir / "labels" / split
        image_output.mkdir(parents=True, exist_ok=True)
        label_output.mkdir(parents=True, exist_ok=True)

        label_names = set()
        class_counts = Counter()
        rle_annotations = 0
        skipped_annotations = 0
        label_rows = 0
        image_items = sorted(images_by_id.items())

        for image_id, image in tqdm(
            image_items, desc=f"Convert {split}", unit="image"
        ):
            source_image = coco_dir / split / image["file_name"]
            target_image = image_output / image["file_name"]
            label_name = f"{Path(image['file_name']).stem}.txt"
            if label_name in label_names:
                raise ValueError(f"중복 label 이름: {split}/{label_name}")
            label_names.add(label_name)

            shutil.copyfile(source_image, target_image)
            lines = []
            for annotation in annotations_by_image.get(image_id, []):
                if annotation.get("iscrowd", 0):
                    continue
                segmentation = annotation.get("segmentation")
                if isinstance(segmentation, dict):
                    rle_annotations += 1
                polygons = segmentation_to_polygons(
                    segmentation, image["height"], image["width"]
                )
                if not polygons:
                    skipped_annotations += 1
                    continue

                class_id = split_mapping[annotation["category_id"]]
                for polygon in polygons:
                    lines.append(
                        polygon_to_yolo_line(
                            class_id, polygon, image["width"], image["height"]
                        )
                    )
                    class_counts[class_names[class_id]] += 1
                    label_rows += 1

            (label_output / label_name).write_text(
                "\n".join(lines) + ("\n" if lines else ""), encoding="utf-8"
            )

        conversion_summary[split] = {
            "images": len(images_by_id),
            "label_rows": label_rows,
            "rle_annotations": rle_annotations,
            "skipped_annotations": skipped_annotations,
            "rows_by_class": dict(sorted(class_counts.items())),
        }

    data = {
        "path": str(yolo_dir),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": class_names,
    }
    data_yaml = yolo_dir / "data.yaml"
    yaml_lines = [
        f"path: {data['path']}",
        f"train: {data['train']}",
        f"val: {data['val']}",
        f"test: {data['test']}",
        "names:",
    ]
    yaml_lines.extend(
        f"  {index}: {json.dumps(name, ensure_ascii=False)}"
        for index, name in enumerate(class_names)
    )
    data_yaml.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")

    summary_path = yolo_dir / "conversion_summary.json"
    summary_path.write_text(
        json.dumps(
            {"class_names": class_names, "splits": conversion_summary},
            ensure_ascii=False,
            indent=2,
        )
        + "\n",
        encoding="utf-8",
    )
    return data_yaml, class_names, conversion_summary


FORCE_REBUILD = False
DATA_YAML, CLASS_NAMES, CONVERSION_SUMMARY = convert_coco_to_yolo(
    COCO_DIR, YOLO_DIR, force_rebuild=FORCE_REBUILD
)
print(DATA_YAML.read_text(encoding="utf-8"))
print(json.dumps(CONVERSION_SUMMARY, ensure_ascii=False, indent=2))


In [ ]:
def validate_yolo_dataset(yolo_dir, class_count):
    checked_images = 0
    checked_labels = 0
    checked_rows = 0

    for split in SPLITS:
        image_dir = yolo_dir / "images" / split
        label_dir = yolo_dir / "labels" / split
        images = sorted(path for path in image_dir.iterdir() if path.is_file())
        labels = sorted(label_dir.glob("*.txt"))
        image_stems = {path.stem for path in images}
        label_stems = {path.stem for path in labels}
        if image_stems != label_stems:
            raise ValueError(
                f"{split}: image/label stem mismatch "
                f"({len(image_stems)} images, {len(label_stems)} labels)"
            )

        for label_path in labels:
            for line_number, line in enumerate(
                label_path.read_text(encoding="utf-8").splitlines(), start=1
            ):
                parts = line.split()
                if len(parts) < 7 or (len(parts) - 1) % 2:
                    raise ValueError(f"잘못된 polygon: {label_path}:{line_number}")
                class_id = int(parts[0])
                coordinates = [float(value) for value in parts[1:]]
                if not 0 <= class_id < class_count:
                    raise ValueError(f"잘못된 class id: {label_path}:{line_number}")
                if not all(0.0 <= value <= 1.0 for value in coordinates):
                    raise ValueError(f"좌표 범위 오류: {label_path}:{line_number}")
                checked_rows += 1

        checked_images += len(images)
        checked_labels += len(labels)

    return {
        "images": checked_images,
        "label_files": checked_labels,
        "polygon_rows": checked_rows,
    }


VALIDATION_SUMMARY = validate_yolo_dataset(YOLO_DIR, len(CLASS_NAMES))
if any(item["skipped_annotations"] for item in CONVERSION_SUMMARY.values()):
    raise ValueError(f"변환하지 못한 annotation이 있습니다: {CONVERSION_SUMMARY}")

print("validation:", VALIDATION_SUMMARY)


## 5. Augmentation 설정

전방 고정 카메라의 도로 구조를 보존하기 위해 상하 반전과 강한 회전/원근 변환은 사용하지 않습니다. 밝기·채도·작은 위치/크기 변화는 적극 사용하고, 수평 반전과 mosaic/copy-paste는 낮은 확률로 적용합니다. 마지막 25 epoch에서는 mosaic를 꺼서 실제 프레임 분포에 맞게 수렴시킵니다.


In [ ]:
AUGMENTATION = {
    "hsv_h": 0.015,
    "hsv_s": 0.50,
    "hsv_v": 0.35,
    "degrees": 3.0,
    "translate": 0.08,
    "scale": 0.25,
    "shear": 1.0,
    "perspective": 0.0002,
    "flipud": 0.0,
    "fliplr": 0.50,
    "mosaic": 0.30,
    "mixup": 0.0,
    "copy_paste": 0.10,
    "copy_paste_mode": "flip",
    "close_mosaic": 25,
}

print(json.dumps(AUGMENTATION, indent=2))


## 6. YOLOv8 segmentation 학습


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-seg.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=300,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    patience=30,
    seed=42,
    deterministic=True,
    amp=True,
    plots=True,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    **AUGMENTATION,
)

BEST_PATH = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
if not BEST_PATH.is_file():
    raise FileNotFoundError(BEST_PATH)
print("best model:", BEST_PATH)


## 7. Test split 평가


In [ ]:
best_model = YOLO(str(BEST_PATH))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    plots=True,
)

print("mask mAP50-95:", metrics.seg.map)
print("mask mAP50:", metrics.seg.map50)


## 8. 모델과 재현 설정 저장


In [ ]:
SAVE_DIR = DRIVE_DIR / "yolo_results" / RUN_NAME
SAVE_DIR.mkdir(parents=True, exist_ok=True)

artifacts = {
    BEST_PATH: SAVE_DIR / "skku_merged_yolov8n_seg_aug_best.pt",
    DATA_YAML: SAVE_DIR / "data.yaml",
}

merge_report = COCO_DIR / "merge_report.json"
if merge_report.is_file():
    artifacts[merge_report] = SAVE_DIR / "merge_report.json"

for name in ("args.yaml", "results.csv"):
    source = RUNS_DIR / RUN_NAME / name
    if source.is_file():
        artifacts[source] = SAVE_DIR / name

for source, target in artifacts.items():
    shutil.copy2(source, target)

(SAVE_DIR / "augmentation.json").write_text(
    json.dumps(AUGMENTATION, indent=2) + "\n", encoding="utf-8"
)

print("saved artifacts:")
for path in sorted(SAVE_DIR.iterdir()):
    print("-", path)
